# DataScribes — End-to-End Pipeline

Ingests dataset metadata from **NYC Open Data** and **Data.gov**, profiles column-level statistics, generates human-readable descriptions using **Claude** via the Anthropic API, and evaluates the results with ROUGE, METEOR, BERTScore, and retrieval NDCG.

**Stages**
1. Ingestion & Sampling — fetch metadata + sample rows from public APIs
2. Profiling — compute per-column stats (type inference, missing counts, examples)
3. LLM Description Generation — build structured prompts → call Claude API
4. Evaluation — automatic, NLP similarity, and retrieval metrics

**Runtime:** ~5-10 minutes end-to-end for the default 10-dataset demo (5 NYC + 5 Data.gov).  
**Requires:** `ANTHROPIC_API_KEY` only for Stage 3.  
**Note:** This notebook is a local demo pipeline. The production 200-dataset run is defined by the scripts in `scripts/` and `evaluation/`.

---
## Setup

In [ ]:
# Install any missing packages (run once; safe to re-run)
import subprocess, sys
pkgs = ["anthropic", "pandas", "pyarrow", "requests", "rouge-score",
        "bert-score", "nltk", "scikit-learn", "openpyxl"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("All packages available.")

In [ ]:
import os

# ── Configuration ─────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")  # set env var or paste key here
CLAUDE_MODEL      = "claude-sonnet-4-5"     # model used in the full DataProc run
NYC_LIMIT         = 5                        # number of NYC Open Data datasets to fetch
DATA_GOV_LIMIT    = 5                        # number of Data.gov datasets to fetch
SAMPLE_ROWS       = 5                        # sample rows fetched per dataset
OUTPUT_DIR        = "data/notebook_output"   # all outputs written here
# ──────────────────────────────────────────────────────────────────────────────

import pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Config OK. Outputs → {OUTPUT_DIR}")
if not ANTHROPIC_API_KEY:
    print("ANTHROPIC_API_KEY is not set. Stages 1-2 and the notebook structure can still be reviewed, but Stage 3 will fail until the key is provided.")

In [ ]:
import json, math, re, time, warnings
from io import StringIO
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import requests
import anthropic

warnings.filterwarnings("ignore")

# NLTK data (for METEOR)
for resource in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

print("Imports OK.")

---
## Stage 1 — Ingestion & Sampling

Fetches dataset metadata from:
- **NYC Open Data** via the Socrata catalog API (`data.cityofnewyork.us/api/views.json`). Sample rows are retrieved with the Socrata JSON resource endpoint (`?$limit=5`) — no full CSV download needed.
- **Data.gov** via the CKAN package search API (`catalog.data.gov/api/3/action/package_search`). Sample rows are obtained by streaming the CSV and stopping after 5 lines.

This notebook keeps the local demo small and readable. The authoritative production pipeline for the 200-dataset run lives in `scripts/dataproc_ingest_and_sample_v2.py` and related batch scripts.

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0"}

# ── NYC Open Data ──────────────────────────────────────────────────────────────

def fetch_nyc_json_sample(dataset_id, limit=5):
    """Fetch sample rows via the Socrata JSON resource API (no CSV download)."""
    url = f"https://data.cityofnewyork.us/resource/{dataset_id}.json?$limit={limit}"
    try:
        r = requests.get(url, timeout=30, headers=HEADERS)
        r.raise_for_status()
        rows = r.json()
        return rows if isinstance(rows, list) else []
    except Exception:
        return []


def fetch_nyc_datasets(limit=NYC_LIMIT):
    print(f"Fetching {limit} NYC Open Data datasets...")
    catalog = requests.get(
        "https://data.cityofnewyork.us/api/views.json",
        timeout=60, headers=HEADERS
    ).json()

    records = []
    for entry in catalog[:limit]:
        dataset_id = entry.get("id")
        if not dataset_id:
            continue

        # Fetch per-dataset detail for column schema
        detail = requests.get(
            f"https://data.cityofnewyork.us/api/views/{dataset_id}.json",
            timeout=30, headers=HEADERS
        ).json()

        columns    = detail.get("columns", [])
        col_names  = [c.get("name", "") for c in columns]
        col_types  = [c.get("dataTypeName", "unknown") for c in columns]
        category   = entry.get("category")
        tags       = entry.get("tags") or []
        keywords   = list(dict.fromkeys(
            ([category] if category else []) + [t for t in tags if t]
        ))
        license_info = detail.get("license")
        license_name = license_info.get("name") if isinstance(license_info, dict) else None

        sample_rows = fetch_nyc_json_sample(dataset_id, limit=SAMPLE_ROWS)

        records.append({
            "dataset_id":         dataset_id,
            "source":             "nyc_open_data",
            "title":              entry.get("name"),
            "original_description": entry.get("description"),
            "keywords_json":      json.dumps(keywords),
            "column_names_json":  json.dumps(col_names),
            "column_types_raw_json": json.dumps(col_types),
            "download_url":       f"https://data.cityofnewyork.us/api/views/{dataset_id}/rows.csv?accessType=DOWNLOAD",
            "landing_page_url":   f"https://data.cityofnewyork.us/d/{dataset_id}",
            "last_updated":       str(entry.get("rowsUpdatedAt", "")),
            "license":            license_name,
            "sample_rows_json":   json.dumps(sample_rows) if sample_rows else None,
        })
        print(f"  ✓ {entry.get('name', dataset_id)[:60]}")

    return records


nyc_records = fetch_nyc_datasets(NYC_LIMIT)
print(f"\nFetched {len(nyc_records)} NYC datasets.")

In [ ]:
# ── Data.gov ───────────────────────────────────────────────────────────────────

def fetch_csv_sample(url, limit=5):
    """Stream a CSV URL and stop after `limit` data rows to avoid full downloads."""
    if not url:
        return [], []
    try:
        r = requests.get(url, timeout=30, stream=True, headers=HEADERS)
        r.raise_for_status()
        lines = []
        for raw in r.iter_lines():
            if raw:
                lines.append(raw.decode("utf-8", errors="replace"))
            if len(lines) >= limit + 1:
                break
        r.close()
        if not lines:
            return [], []
        df = pd.read_csv(StringIO("\n".join(lines)))
        return df.to_dict(orient="records"), list(df.columns)
    except Exception:
        return [], []


def fetch_datagov_datasets(limit=DATA_GOV_LIMIT):
    print(f"Fetching {limit} Data.gov datasets via CKAN API...")
    resp = requests.get(
        "https://catalog.data.gov/api/3/action/package_search",
        params={"rows": limit, "start": 0},
        timeout=60, headers=HEADERS
    )
    resp.raise_for_status()
    results = resp.json()["result"]["results"]

    records = []
    for entry in results:
        dataset_id = entry.get("id") or entry.get("name")
        if not dataset_id:
            continue

        # Find CSV resource URL
        csv_url = None
        for resource in (entry.get("resources") or []):
            fmt = (resource.get("format") or "").lower()
            url  = resource.get("url", "")
            if "csv" in fmt or url.lower().endswith(".csv"):
                csv_url = url
                break

        tags     = [t.get("display_name") or t.get("name", "") for t in (entry.get("tags") or [])]
        sample_rows, sample_cols = fetch_csv_sample(csv_url, limit=SAMPLE_ROWS)

        records.append({
            "dataset_id":         str(dataset_id),
            "source":             "data_gov",
            "title":              entry.get("title"),
            "original_description": entry.get("notes"),
            "keywords_json":      json.dumps(tags),
            "column_names_json":  json.dumps(sample_cols),
            "column_types_raw_json": json.dumps([]),
            "download_url":       csv_url,
            "landing_page_url":   entry.get("url"),
            "last_updated":       entry.get("metadata_modified"),
            "license":            entry.get("license_title"),
            "sample_rows_json":   json.dumps(sample_rows) if sample_rows else None,
        })
        print(f"  ✓ {str(entry.get('title', dataset_id))[:60]}")

    return records


datagov_records = fetch_datagov_datasets(DATA_GOV_LIMIT)
print(f"\nFetched {len(datagov_records)} Data.gov datasets.")

In [ ]:
# ── Combine and save ──────────────────────────────────────────────────────────
metadata_df = pd.DataFrame(nyc_records + datagov_records)
metadata_df.to_parquet(f"{OUTPUT_DIR}/metadata.parquet", index=False)

print(f"Combined: {len(metadata_df)} datasets ({len(nyc_records)} NYC + {len(datagov_records)} Data.gov)")
metadata_df[["source", "title", "dataset_id"]].head(10)

---
## Stage 2 — Profiling

For each dataset, analyzes the sample rows fetched in Stage 1 to compute per-column statistics:
- **Inferred type** — `numeric_like`, `date_like`, or `text_like` based on whether all values parse as floats or dates
- **Non-null / missing counts** and **unique count** in the sample
- **Example values** (up to 3) used later in the LLM prompt

In [ ]:
def infer_type(values):
    cleaned = [v for v in values if v is not None and str(v).strip()]
    if not cleaned:
        return "unknown"
    strs = [str(v).strip() for v in cleaned]
    if all(_is_numeric(v) for v in strs):
        return "numeric_like"
    if all(_is_date(v) for v in strs):
        return "date_like"
    return "text_like"


def _is_numeric(s):
    try:
        float(s); return True
    except Exception:
        return False


def _is_date(s):
    return not pd.isna(pd.to_datetime(s, errors="coerce"))


def profile_sample_rows(sample_rows_json):
    """Returns a dict mapping column_name → stats dict."""
    if not sample_rows_json or pd.isna(sample_rows_json):
        return {}
    try:
        rows = json.loads(sample_rows_json)
    except Exception:
        return {}
    if not rows:
        return {}

    df = pd.DataFrame(rows)
    profile = {}
    for col in df.columns:
        vals = df[col].tolist()
        profile[col] = {
            "non_null_count":      int(df[col].notna().sum()),
            "missing_count":       int(df[col].isna().sum()),
            "unique_count_sample": int(df[col].nunique(dropna=True)),
            "inferred_sample_type": infer_type(vals),
            "example_values":      df[col].dropna().astype(str).head(3).tolist(),
        }
    return profile


print("Profiling functions defined.")

In [ ]:
profile_rows = []
for _, row in metadata_df.iterrows():
    col_names = json.loads(row["column_names_json"] or "[]")
    col_types = json.loads(row["column_types_raw_json"] or "[]")
    sp = profile_sample_rows(row["sample_rows_json"])

    profile_rows.append({
        "dataset_id":                 row["dataset_id"],
        "source":                     row["source"],
        "title":                      row["title"],
        "original_description":       row["original_description"],
        "keywords_json":              row["keywords_json"],
        "column_names_json":          json.dumps(col_names),
        "column_types_raw_json":      json.dumps(col_types),
        "sample_rows_json":           row["sample_rows_json"],
        "sample_columns_profile_json": json.dumps(sp),
        "declared_column_count":      len(col_names),
        "sample_row_count":           len(json.loads(row["sample_rows_json"] or "[]")),
        "sample_column_count":        len(sp),
    })

profiles_df = pd.DataFrame(profile_rows)
profiles_df.to_parquet(f"{OUTPUT_DIR}/profiles.parquet", index=False)

print(f"Profiled {len(profiles_df)} datasets.")
profiles_df[["source", "title", "declared_column_count", "sample_row_count", "sample_column_count"]].head(10)

---
## Stage 3 — LLM Description Generation

Calls the Claude API for each dataset using a structured prompt built from:
- Dataset title and source
- Original description (capped at 500 chars to stay within token limits)
- Up to 8 keywords
- Up to 12 column names with total count
- Column-level profiles (inferred type + 2 example values) for up to 6 columns
- One sample row as JSON (capped at 300 chars)

The prompt instructs Claude to write 3-5 sentences that are specific, informative, and not a verbatim copy of the original.

In [ ]:
def build_prompt(row) -> str:
    title               = row.get("title") or "Untitled Dataset"
    source              = row.get("source") or ""
    original_desc       = (row.get("original_description") or "").strip()[:500]
    keywords            = json.loads(row.get("keywords_json") or "[]")
    column_names        = json.loads(row.get("column_names_json") or "[]")
    sample_rows         = json.loads(row.get("sample_rows_json") or "[]")
    sample_profile      = json.loads(row.get("sample_columns_profile_json") or "{}")

    lines = [
        f"Dataset Title: {title}",
        f"Source: {source}",
    ]
    if original_desc:
        lines.append(f"Original Description: {original_desc}")
    if keywords:
        lines.append(f"Keywords: {', '.join(keywords[:8])}")
    if column_names:
        lines.append(f"Columns ({len(column_names)} total): {', '.join(column_names[:12])}")
    if sample_profile:
        profile_lines = []
        for col_name, stats in list(sample_profile.items())[:6]:
            examples  = stats.get("example_values", [])[:2]
            inferred  = stats.get("inferred_sample_type", "")
            profile_lines.append(f"  - {col_name} ({inferred}): e.g. {', '.join(str(v) for v in examples)}")
        if profile_lines:
            lines.append("Column Profiles:\n" + "\n".join(profile_lines))
    if sample_rows and isinstance(sample_rows, list):
        lines.append(f"Sample row: {json.dumps(sample_rows[0], ensure_ascii=False)[:300]}")

    context = "\n".join(lines)
    return (
        "You are a data catalog assistant. Write a clear, concise description "
        "(3-5 sentences) for the following dataset.\n"
        "The description should help a data analyst quickly understand what the "
        "dataset contains, what it can be used for, and any notable characteristics.\n"
        "Do not copy the original description verbatim. Be specific and informative.\n"
        f"\n{context}\n\nDescription:"
    )


# Preview the prompt for the first dataset
sample_row = profiles_df.iloc[0].to_dict()
print(build_prompt(sample_row))

In [ ]:
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

output_rows = []
for i, row in profiles_df.iterrows():
    title = row.get("title", row["dataset_id"])
    print(f"[{i+1}/{len(profiles_df)}] {str(title)[:60]}...", end=" ")
    try:
        prompt  = build_prompt(row.to_dict())
        message = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
        )
        description = message.content[0].text.strip()
        error       = None
        print("OK")
    except Exception as e:
        description = None
        error       = str(e)
        print(f"FAILED ({error[:60]})")
        time.sleep(2)

    output_rows.append({
        "dataset_id":           row["dataset_id"],
        "source":               row["source"],
        "title":                row["title"],
        "original_description": row["original_description"],
        "generated_description": description,
        "generation_model":     CLAUDE_MODEL,
        "generation_error":     error,
    })

descriptions_df = pd.DataFrame(output_rows)
descriptions_df.to_parquet(f"{OUTPUT_DIR}/descriptions.parquet", index=False)
descriptions_df.to_csv(f"{OUTPUT_DIR}/descriptions.csv", index=False)

n_ok  = descriptions_df["generated_description"].notna().sum()
n_tot = len(descriptions_df)
print(f"\nGenerated {n_ok}/{n_tot} descriptions ({100*n_ok/n_tot:.0f}% success).")

In [ ]:
# Show a side-by-side comparison for the first successful row
first_ok = descriptions_df[descriptions_df["generated_description"].notna()].iloc[0]

print("=" * 70)
print(f"DATASET: {first_ok['title']}  ({first_ok['source']})")
print("=" * 70)
print("\n— ORIGINAL —")
print(str(first_ok["original_description"])[:500])
print("\n— GENERATED —")
print(first_ok["generated_description"])

---
## Stage 4 — Evaluation

Three evaluation categories matching the project proposal:

| Category | Metrics |
|---|---|
| **Automatic** | Success rate, word/sentence counts, novelty score, title coverage |
| **Text Similarity** | ROUGE-1/2/L, METEOR, BERTScore F1 |
| **Retrieval** | TF-IDF cosine ranking + NDCG@5 and NDCG@10 over 8 realistic queries |

The original human-written description is treated as the reference for all NLP metrics.

In [ ]:
# ── 4a. Automatic Metrics ─────────────────────────────────────────────────────

STOPWORDS = {"the", "a", "an", "of", "in", "and", "or", "for", "to", "by", "on", "at", "with"}

def word_set(text):
    if not text or (isinstance(text, float) and math.isnan(text)):
        return set()
    return set(re.findall(r"[a-z]+", str(text).lower()))

def word_count(text):
    if not text or (isinstance(text, float) and math.isnan(text)):
        return 0
    return len(str(text).split())

def sentence_count(text):
    if not text or (isinstance(text, float) and math.isnan(text)):
        return 0
    return len(re.findall(r"[.!?]+", str(text).strip()))

def novelty_score(generated, original):
    gen  = word_set(generated)
    orig = word_set(original)
    if not gen:
        return 0.0
    return round(len(gen - orig) / len(gen), 3)

def title_coverage(generated, title):
    if not generated or not title:
        return False
    title_words = {w for w in word_set(title) if w not in STOPWORDS and len(w) > 2}
    return bool(title_words & word_set(generated))


eval_df = descriptions_df.copy()
eval_df["success"]              = eval_df["generated_description"].notna()
eval_df["orig_word_count"]      = eval_df["original_description"].apply(word_count)
eval_df["gen_word_count"]       = eval_df["generated_description"].apply(word_count)
eval_df["gen_sentence_count"]   = eval_df["generated_description"].apply(sentence_count)
eval_df["follows_3_5_sentences"]= eval_df["gen_sentence_count"].between(3, 5)
eval_df["novelty_score"]        = eval_df.apply(
    lambda r: novelty_score(r["generated_description"], r["original_description"]), axis=1
)
eval_df["title_coverage"]       = eval_df.apply(
    lambda r: title_coverage(r["generated_description"], r["title"]), axis=1
)

auto_cols = ["source", "title", "success", "orig_word_count", "gen_word_count",
             "gen_sentence_count", "follows_3_5_sentences", "novelty_score", "title_coverage"]
eval_df[auto_cols].head(10)

In [ ]:
success = eval_df[eval_df["success"]]
print(f"Success rate  : {eval_df['success'].mean()*100:.1f}%")
print(f"Avg words     : {success['gen_word_count'].mean():.1f} generated  /  {eval_df['orig_word_count'].mean():.1f} original")
print(f"Avg sentences : {success['gen_sentence_count'].mean():.2f}")
print(f"Follows 3-5s  : {success['follows_3_5_sentences'].mean()*100:.1f}%")
print(f"Novelty score : {success['novelty_score'].mean():.3f}  (1.0 = all-new vocab)")
print(f"Title coverage: {success['title_coverage'].mean()*100:.1f}%")

In [ ]:
# ── 4b. Text Similarity — ROUGE + METEOR ─────────────────────────────────────
from rouge_score import rouge_scorer as rouge_lib
from nltk.translate import meteor_score as meteor_lib
from nltk.tokenize import word_tokenize

scorer = rouge_lib.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

scorable = eval_df[
    eval_df["generated_description"].notna() &
    eval_df["original_description"].notna() &
    (eval_df["original_description"].str.strip() != "")
].copy()

r1, r2, rL, meteors = [], [], [], []
for _, row in scorable.iterrows():
    ref = str(row["original_description"]).strip()
    hyp = str(row["generated_description"]).strip()

    scores = scorer.score(ref, hyp)
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

    try:
        m = meteor_lib.meteor_score([word_tokenize(ref.lower())], word_tokenize(hyp.lower()))
    except Exception:
        m = 0.0
    meteors.append(m)

scorable["rouge1"]  = r1
scorable["rouge2"]  = r2
scorable["rougeL"]  = rL
scorable["meteor"]  = meteors

print(f"ROUGE-1 : {scorable['rouge1'].mean():.4f}")
print(f"ROUGE-2 : {scorable['rouge2'].mean():.4f}")
print(f"ROUGE-L : {scorable['rougeL'].mean():.4f}")
print(f"METEOR  : {scorable['meteor'].mean():.4f}")

In [ ]:
# ── BERTScore (downloads ~400 MB roberta-large model on first run) ────────────
from bert_score import score as bertscore_fn

refs = scorable["original_description"].tolist()
hyps = scorable["generated_description"].tolist()

P, R, F1 = bertscore_fn(hyps, refs, lang="en", verbose=True)
scorable["bertscore_p"]  = P.numpy()
scorable["bertscore_r"]  = R.numpy()
scorable["bertscore_f1"] = F1.numpy()

sim_out_cols = ["dataset_id", "source", "title",
                "rouge1", "rouge2", "rougeL", "meteor",
                "bertscore_p", "bertscore_r", "bertscore_f1"]
similarity_df = scorable[sim_out_cols].round(4)
similarity_df.to_csv(f"{OUTPUT_DIR}/text_similarity.csv", index=False)

print(f"BERTScore F1 : {similarity_df['bertscore_f1'].mean():.4f}")
similarity_df.head(10)

In [ ]:
# ── 4c. Retrieval Evaluation — TF-IDF + NDCG ─────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

QUERIES = [
    ("NYC taxi trip data",           ["nyc", "taxi", "trip", "cab", "fare", "ride"]),
    ("weather wind speed dataset",   ["weather", "wind", "speed", "temperature", "climate", "forecast"]),
    ("housing property values",      ["housing", "property", "real estate", "home", "price", "value", "zoning"]),
    ("crime incident reports",       ["crime", "incident", "arrest", "complaint", "police", "felony"]),
    ("public health disease data",   ["health", "disease", "hospital", "covid", "vaccination", "death"]),
    ("school education enrollment",  ["school", "education", "student", "enrollment", "grade", "teacher"]),
    ("traffic collision accidents",  ["traffic", "collision", "accident", "crash", "vehicle", "road"]),
    ("restaurant food inspection",   ["restaurant", "food", "inspection", "violation", "health", "permit"]),
]


def dcg(rels): return sum(r / math.log2(i + 2) for i, r in enumerate(rels))
def ndcg(rels): ideal = sorted(rels, reverse=True); d = dcg(ideal); return dcg(rels) / d if d else 0.0
def kw_rel(text, keywords): return int(any(kw in text.lower() for kw in keywords))


def rank_corpus(query, corpus):
    vec = TfidfVectorizer(stop_words="english")
    try:
        mat = vec.fit_transform(corpus + [query])
    except ValueError:
        return list(range(len(corpus)))
    sims = cos_sim(mat[-1], mat[:-1]).flatten()
    return sorted(range(len(corpus)), key=lambda i: sims[i], reverse=True)


ok = descriptions_df[descriptions_df["generated_description"].notna()].copy()
orig_corpus = [
    f"{r['title'] or ''} {r['original_description'] or ''}"
    for _, r in ok.iterrows()
]
gen_corpus = [
    f"{r['title'] or ''} {r['generated_description'] or ''}"
    for _, r in ok.iterrows()
]

ret_records = []
for query, keywords in QUERIES:
    orig_ranked = rank_corpus(query, orig_corpus)
    gen_ranked  = rank_corpus(query, gen_corpus)
    for k in [5, 10]:
        orig_rels = [kw_rel(orig_corpus[i], keywords) for i in orig_ranked[:k]]
        gen_rels  = [kw_rel(gen_corpus[i],  keywords) for i in gen_ranked[:k]]
        ret_records.append({
            "query":          query,
            "k":              k,
            "ndcg_original":  round(ndcg(orig_rels), 4),
            "ndcg_generated": round(ndcg(gen_rels),  4),
            "delta":          round(ndcg(gen_rels) - ndcg(orig_rels), 4),
        })

retrieval_df = pd.DataFrame(ret_records)
retrieval_df.to_csv(f"{OUTPUT_DIR}/retrieval_results.csv", index=False)
retrieval_df[retrieval_df["k"] == 10]

---
## Final Summary

In [ ]:
print("=" * 65)
print("DATASCRIBES PIPELINE — RESULTS SUMMARY")
print("=" * 65)

total   = len(eval_df)
n_ok    = int(eval_df["success"].sum())
success = eval_df[eval_df["success"]]

print(f"\n Datasets processed : {total}")
print(f" Successful         : {n_ok} ({100*n_ok/total:.1f}%)")
print(f" Failed             : {total - n_ok}")
print(f" Model              : {CLAUDE_MODEL}")

print("\n— Automatic Metrics —")
print(f" Avg generated words    : {success['gen_word_count'].mean():.1f}")
print(f" Avg generated sentences: {success['gen_sentence_count'].mean():.2f}")
print(f" Follows 3-5 sentences  : {success['follows_3_5_sentences'].mean()*100:.1f}%")
print(f" Novelty score          : {success['novelty_score'].mean():.3f}")
print(f" Title coverage         : {success['title_coverage'].mean()*100:.1f}%")

print("\n— Text Similarity (vs. original) —")
if "rouge1" in similarity_df.columns:
    print(f" ROUGE-1     : {similarity_df['rouge1'].mean():.4f}")
    print(f" ROUGE-2     : {similarity_df['rouge2'].mean():.4f}")
    print(f" ROUGE-L     : {similarity_df['rougeL'].mean():.4f}")
    print(f" METEOR      : {similarity_df['meteor'].mean():.4f}")
    print(f" BERTScore F1: {similarity_df['bertscore_f1'].mean():.4f}")

print("\n— Retrieval NDCG@10 —")
at10 = retrieval_df[retrieval_df["k"] == 10]
print(f" Original  : {at10['ndcg_original'].mean():.4f}")
print(f" Generated : {at10['ndcg_generated'].mean():.4f}")
print(f" Delta     : {at10['delta'].mean():+.4f}")

print(f"\n Outputs saved to: {OUTPUT_DIR}/")
print("=" * 65)

---
## Output Files

| File | Contents |
|---|---|
| `data/notebook_output/metadata.parquet` | Raw metadata + sample rows (Stage 1) |
| `data/notebook_output/profiles.parquet` | Column-level profiling stats (Stage 2) |
| `data/notebook_output/descriptions.parquet` | Generated descriptions + errors (Stage 3) |
| `data/notebook_output/descriptions.csv` | Same as above in CSV format |
| `data/notebook_output/text_similarity.csv` | ROUGE / METEOR / BERTScore per dataset (Stage 4) |
| `data/notebook_output/retrieval_results.csv` | NDCG@5 and NDCG@10 per query (Stage 4) |

To scale to 200 datasets (the full production run) submit the DataProc PySpark scripts in `scripts/` — see the README for `gcloud dataproc jobs submit` commands.